# Multi-Label Classification (ChestMNIST)

A first **multi-label** study: each 28x28 chest X-ray can carry several findings at once, so the target is 14 independent binary flags (the NIH ChestX-ray14 taxonomy), not one class. The head is 14 sigmoids with binary cross-entropy, and we score with **per-class AUC** (accuracy is meaningless when most labels are 0). This is the multi-label sibling of [pneumonia_mnist](../pneumonia_mnist/); it is a scaffold to build on.

## Setup

In [ ]:
# One-time setup: make the `visualization` helper importable, then fetch data +
# resolve paths. Each study's fetch logic lives in its own download_data.py.
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("ConvolutedComputerVision"):
        !git clone -q https://github.com/samlowe106/ConvolutedComputerVision.git
    sys.path.insert(0, "ConvolutedComputerVision/src")

from visualization import colab_bootstrap

DATA_ROOT, CKPT_ROOT = colab_bootstrap(study="chest-mnist")

In [ ]:
import datetime

import numpy as np
import tensorflow as tf
from sklearn.metrics import roc_auc_score

from visualization import label_pos_weights

notebook_start_time = datetime.datetime.now()

## Data: 28x28 images, 14 binary labels

In [ ]:
LABELS = [
    "atelectasis",
    "cardiomegaly",
    "effusion",
    "infiltration",
    "mass",
    "nodule",
    "pneumonia",
    "pneumothorax",
    "consolidation",
    "edema",
    "emphysema",
    "fibrosis",
    "pleural_thickening",
    "hernia",
]
_npz = np.load(os.path.join(DATA_ROOT, "chestmnist.npz"))


def _make_ds(split):
    images = _npz[f"{split}_images"][..., None]  # (N, 28, 28, 1) uint8
    labels = _npz[f"{split}_labels"].astype("float32")  # (N, 14) in {0, 1}
    return tf.data.Dataset.from_tensor_slices((images, labels))


batch_size = 128
train_ds = _make_ds("train").shuffle(4000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
validation_ds = _make_ds("val").batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_ds = _make_ds("test").batch(batch_size).prefetch(tf.data.AUTOTUNE)
y_test = np.concatenate([y for _, y in test_ds], axis=0)
print("labels per split:", _npz["train_labels"].shape, _npz["test_labels"].shape)

## Model: a small CNN with 14 sigmoid outputs

Each finding is rare, so we train with a per-label weighted binary cross-entropy: `label_pos_weights` gives `n_neg / n_pos` per label (cached), applied to the positive term.

In [ ]:
def build_cnn():
    return tf.keras.models.Sequential(
        [
            tf.keras.layers.Input((28, 28, 1)),
            tf.keras.layers.Rescaling(1.0 / 255),
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(14, activation="sigmoid", name="output"),
        ],
        name="chest_mnist_cnn",
    )


model = build_cnn()

# multi-label imbalance: each finding's positive is rare, so up-weight it. label_pos_weights
# gives n_neg/n_pos per label (cached); we apply it to the positive term of a weighted BCE.
pos_weights = tf.constant(label_pos_weights(_npz["train_labels"], cache_dir=DATA_ROOT))


def weighted_bce(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    losses = -(
        pos_weights * y_true * tf.math.log(y_pred)
        + (1 - y_true) * tf.math.log(1 - y_pred)
    )
    return tf.reduce_mean(losses)


model.compile(
    optimizer=tf.keras.optimizers.AdamW(1e-3),
    loss=weighted_bce,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")],
)
model.summary()

## Train (baseline) and next steps

A short run, then per-class AUC on test. To go further: try focal loss on top of the per-label weighted BCE used here, report per-class precision/recall at chosen operating points, and try a stronger backbone. Grad-CAM would need a per-label target since there are 14 outputs.

In [ ]:
history = model.fit(train_ds, validation_data=validation_ds, epochs=10)

probs = model.predict(test_ds, verbose=0)
print("\nPer-class ROC-AUC (test):")
for i, name in enumerate(LABELS):
    col = y_test[:, i]
    auc = roc_auc_score(col, probs[:, i]) if col.min() != col.max() else float("nan")
    print(f"  {name:<20} {auc:.3f}")
print(
    f"  {'MACRO':<20} {np.nanmean([roc_auc_score(y_test[:, i], probs[:, i]) for i in range(14)]):.3f}"
)

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} "
    f"(duration: {notebook_end_time - notebook_start_time})"
)